In [ ]:
import kagglehub
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
        print(os.path.join(dirname))

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout,Convolution2D,MaxPooling2D,Flatten #action detectionimport tensorflow
import tensorflow as tf
import matplotlib.pyplot as plt
from IPython.display import HTML

In [ ]:
import kagglehub
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# Download latest version of the dataset
path = kagglehub.dataset_download("kshitij192/cars-image-dataset")
print("Path to dataset files:", path)

IMAGE_SIZE = 128

# Define the training path (assuming the download contains a 'Cars Dataset/train' folder)
train_path = os.path.join(path, 'Cars Dataset', 'train')

train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=10,
        horizontal_flip=True
)

train_generator = train_datagen.flow_from_directory(
        train_path,
        target_size=(IMAGE_SIZE, IMAGE_SIZE),
        class_mode="sparse"
)

In [ ]:
import matplotlib.pyplot as plt
import os
import random
from PIL import Image

# Define dataset_root and nama_kelas
dataset_root = os.path.join(path, 'Cars Dataset', 'train') # Assuming you want to display from the training set
nama_kelas = list(train_generator.class_indices.keys())

for class_name in nama_kelas:
    class_path = os.path.join(dataset_root, class_name)
    images = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]
    random.shuffle(images)

    print(f"Display 5 sample dari kelas: '{class_name}'")
    plt.figure(figsize=(15, 3))
    for i in range(min(5, len(images))):
        img_path = os.path.join(class_path, images[i])
        img = Image.open(img_path)
        plt.subplot(1, 5, i + 1)
        plt.imshow(img)
        plt.title(class_name)
        plt.axis('off')
    plt.show()


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define image dimensions and batch size
img_height = 224
img_width = 224
batch_size = 32

# Data augmentation and preprocessing for training data
train_datagen = ImageDataGenerator(
    rescale=1./255,          # Normalize pixel values to [0, 1]
    rotation_range=20,       # Rotate images by up to 20 degrees
    horizontal_flip=True,    # Randomly flip images horizontally
    validation_split=0.2     # Split 20% of data for validation
)

# Only rescale for validation data (no augmentation)
val_datagen = ImageDataGenerator(
    rescale=1./255,          # Normalize pixel values to [0, 1]
    validation_split=0.2     # Use the same split for validation
)

# Load training data from directory
train_generator = train_datagen.flow_from_directory(
    dataset_root,            # Path to your dataset root directory
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical', # Use 'categorical' for multi-class classification
    subset='training',        # Specify this is the training subset
    seed=42                   # For reproducible results
)

# Load validation data from directory
validation_generator = val_datagen.flow_from_directory(
    dataset_root,            # Path to your dataset root directory
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical', # Use 'categorical' for multi-class classification
    subset='validation',      # Specify this is the validation subset
    seed=42                   # For reproducible results
)

print(f"Number of training samples: {train_generator.samples}")
print(f"Number of validation samples: {validation_generator.samples}")
print(f"Classes: {train_generator.class_indices}")


In [ ]:
count=0
for image_batch, label_batch in train_generator:
#     print(label_batch)
    print(image_batch[0])
    break

In [ ]:
class_names = list(train_generator.class_indices.keys())
class_names

In [ ]:
test_path = os.path.join(path, 'Cars Dataset', 'test')

test_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=10,
        horizontal_flip=True)

test_generator = test_datagen.flow_from_directory(
        test_path,
        target_size=(IMAGE_SIZE, IMAGE_SIZE),
        class_mode="sparse"
)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

# Get the number of classes from the generator
num_classes = train_generator.num_classes

# Load the pre-trained MobileNetV2 model without the top classification layer
base_model = MobileNetV2(input_shape=(img_height, img_width, 3),
                          include_top=False,
                          weights='imagenet')

# Freeze the base model layers so they are not updated during training
for layer in base_model.layers:
    layer.trainable = False

# Add a new classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x) # Additional dense layer
predictions = Dense(num_classes, activation='softmax')(x) # Output layer with softmax for multi-class classification

# Create the new model
model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

In [ ]:
print("Train samples:", train_generator.samples)
print("Train batch size:", train_generator.batch_size)
print("Train steps:", len(train_generator))

print("Validation samples:", validation_generator.samples)
print("Validation batch size:", validation_generator.batch_size)
print("Validation steps:", len(validation_generator))

In [ ]:
import os

# Using train_path which was defined in a previous cell
for root, dirs, files in os.walk(train_path):
    # Only print directories that actually contain files
    if files:
        print(f"{root}: {len(files)} files")

In [ ]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=10
)

In [ ]:
import matplotlib.pyplot as plt

# Plot training and validation accuracy
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot training and validation loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
loss, accuracy = model.evaluate(validation_generator)
print(f"Validation Loss: {loss:.4f}")
print(f"Validation Accuracy: {accuracy:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image

# Get a batch of images and labels from the validation generator
x_val, y_val = next(validation_generator)

# Get class names from the generator
class_names = list(validation_generator.class_indices.keys())

plt.figure(figsize=(15, 8))

# Display a few samples and their predictions
for i in range(5):
    img_array = x_val[i]
    true_label_idx = np.argmax(y_val[i])
    true_label = class_names[true_label_idx]

    # Make prediction
    prediction = model.predict(np.expand_dims(img_array, axis=0))
    predicted_label_idx = np.argmax(prediction)
    predicted_label = class_names[predicted_label_idx]

    plt.subplot(1, 5, i + 1)
    plt.imshow(img_array)
    plt.title(f"True: {true_label}\nPred: {predicted_label}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Reset the validation generator
validation_generator.reset()

# Initialize lists to store true labels and predictions
all_true_labels = []
all_predictions = []

# Iterate through the validation generator to collect true labels and make predictions
num_validation_samples = validation_generator.samples
batch_size = validation_generator.batch_size
steps_per_epoch = np.ceil(num_validation_samples / batch_size)

for i in range(int(steps_per_epoch)):
    images_batch, labels_batch = next(validation_generator)

    # Store true labels (convert from one-hot encoding if necessary)
    all_true_labels.extend(np.argmax(labels_batch, axis=1))

    # Get model predictions for the current batch
    predictions_batch = model.predict(images_batch)
    all_predictions.extend(np.argmax(predictions_batch, axis=1))

# Convert lists to numpy arrays
y_true_aligned = np.array(all_true_labels)
y_pred_aligned = np.array(all_predictions)

# Get class names from the generator in the correct order
class_labels = list(validation_generator.class_indices.keys())

print("Corrected Classification Report:")
print(classification_report(y_true_aligned, y_pred_aligned, target_names=class_labels))

# Generate Confusion Matrix
cm_aligned = confusion_matrix(y_true_aligned, y_pred_aligned)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_aligned, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
plt.title('Corrected Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
print(class_names)
print(model.output_shape)

In [ ]:
import gradio as gr
import numpy as np
from PIL import Image

# Ambil nama kelas SEKALI di luar fungsi
class_names = list(validation_generator.class_indices.keys())

print("Classes:", class_names)

def predict_image(image):

    if image is None:
        return {"Upload gambar terlebih dahulu": 1.0}

    image = Image.fromarray(image.astype(np.uint8))
    image = image.resize((img_width, img_height))

    image_array = np.asarray(image) / 255.0
    image_array = np.expand_dims(image_array, axis=0)

    predictions = model.predict(image_array, verbose=0)
    probabilities = predictions[0]

    result = {
        class_names[i]: float(probabilities[i])
        for i in range(len(class_names))
    }

    return result

iface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="numpy", label="Upload Image"),
    outputs=gr.Label(num_top_classes=3),
    title="Car Classification",
    description="Upload an image of a car to classify."
)

iface.launch(debug=True, share=True)